In [4]:
#Imports
import pandas as pd
import requests 
import bs4

Provide the series id (including 'tt') and the last season number.<br>
verbose is an integer with the usual meaning.(Values can be either 0 or 1) by defaualt it is 1.<br>
returns a pandas DataFrame
#### eg. make_database(series_id = 'tt0056751', last_season = 26).<br>


In [2]:
def tv_series(series_id = None, last_season = 0, verbose = 1):
    
    """
    Provide the series id (including 'tt') and the last season number.
    eg. make_database(series_id = 'tt0056751', last_season = 26).
    verbose is an integer with the usual meaning.(Values can be either 0 or 1) by defaualt it is 1.
    returns a pandas DataFrame
    """
    
    #Imports 
    import pandas as pd
    import requests 
    import bs4
    
    
    #Containers
    ep_names = []
    ep_numbers = []
    air_dates = []
    ratings = []
    num_ratings = []
    descriptions = []
    seasons = []
    
    #No last Season provided
    if last_season == 0:
        return None
    
    
    #Starting the loop
    s=1
    while True:
        
        #Setting the request and genrating the soup
        link = f"https://www.imdb.com/title/{series_id}/episodes?season={s}"
        code = requests.get(link)
        soup = bs4.BeautifulSoup(code.text, "lxml")

        #Printing the current season to show the progress
        c_season = soup.select('#episode_top')[0].text
        if verbose:
            print(c_season)

        #Starting the For loop
        for i in range(len(soup.select('.info'))):
            #Name of Episode
            episode = soup.select('strong')[i].text
            ep_names.append(episode)
            
            #Number of Episode
            if len(str(s)) == 1 and i<9:
                ep_num = f"S0{s}E0{i+1}"
                
            elif len(str(s)) == 2 and i<9:
                ep_num = f"S{s}E0{i+1}"
                
            elif len(str(s)) == 1 and i>8:
                ep_num = f"S0{s}E{i+1}"
                
            elif len(str(s)) == 2 and i>8:
                ep_num = f"S{s}E{i+1}"
                
            ep_numbers.append(ep_num)

            #Date Aired
            date = soup.select('.airdate')[i].text.strip()
            air_dates.append(date)

            #Average Rating
            rating = soup.select('.ipl-rating-star__rating')[23*i].text
            ratings.append(rating)

            #Number of ratings
            rating_numlist = list(soup.select('.ipl-rating-star__total-votes')[i].text)
            rating_numlist.remove('(')
            rating_numlist.remove(')')
            num_rating = ''.join(rating_numlist)
            num_ratings.append(num_rating)

            #Decscription
            description = soup.select('.item_description')[i].text.strip()
            descriptions.append(description)

            #Season
            season = soup.select('#episode_top')[0].text
            seasons.append(' '.join(season.split()))


        #Checking if it is final season then breaking the loop
        
        if s==last_season:
            break
        else:
            #Next Page
            s=s+1
    
    cols = 'Season, Name of Episode, ID of the Episode, Date Aired, Average Rating, Number of Ratings, Description'.split(', ')
    return pd.DataFrame(data = [seasons, ep_names, ep_numbers, air_dates, ratings, num_ratings, descriptions], index=cols).transpose()

Scarping book info from Goodreads Users's page <br>
the user_code is 9 digit code found in the link of user's 'My Books' page <br>
If no shelf name is provided, info from 'All' shelf is returned <br>
verbose has usual meaning with value being integer 0 or 1, default is 1. <br>
#### eg. book_from_goodreads(user_code =110673662, shelf_name = 'currently-reading', verbose = 1)

In [3]:
def user_books_from_goodreads(user_code = None, shelf_name = None, verbose = 1):
    """
    Scarping book info from Goodreads Users's page
    the user_code is 9 digit code found in the link of user's 'My Books' page
    If no shelf name is provided, info from 'All' shelf is returned
    verbose has usual meaning with value being integer 0 or 1, default is 1.
    eg. book_from_goodreads(user_code =110673662, shelf_name = 'currently-reading', verbose = 1)
    """

    #Scarping book info from Goodreads

    #Imports
    import requests
    import bs4
    import pandas as pd

    #Containers
    books = []
    authors = []
    isbn13s = []
    isbns =[]
    pagess = []
    ratings = []
    num_ratings = []
    pub_dates = []
    date_started = []
    date_completed = []

    p=1
    while True:

        if not shelf_name:
            link = f'https://www.goodreads.com/review/list/110673662?page={p}'
        else:
            link = f'https://www.goodreads.com/review/list/110673662?page={p}&ref=nav_mybooks&shelf={shelf_name}'

        code = requests.get(link)
        soup = bs4.BeautifulSoup(code.text, "lxml")

        #Breaking the loop
        if soup.select('.bookalike') == []:
            break

        if verbose == 0:
            pass 
        else:
            print(p)

        for i in range(len(soup.select('.bookalike'))):

            book_name = soup.select('.bookalike .title a')[i]['title']
            books.append(book_name)

            author = soup.select('.bookalike .author a')[i].text
            authors.append(author)

            isbn13 = soup.select('.bookalike .isbn13 .value')[i].text.strip()
            isbn13s.append(isbn13)
            isbn = soup.select('.bookalike .isbn .value')[i].text.strip()
            isbns.append(isbn)

            pages = soup.select('.bookalike .num_pages .value')[i].text.split('\n')[1].strip()
            pagess.append(pages)

            rating = float(soup.select('.bookalike .avg_rating .value')[i].text.strip())
            ratings.append(rating)

            num = soup.select('.bookalike .num_ratings .value')[i].text.strip()
            num_rating = int(''.join(num.split(',')))
            num_ratings.append(num_rating)

            pub_date = soup.select('.bookalike .date_pub .value')[i].text.strip()
            pub_dates.append(pub_date)
            
            started = soup.select('.bookalike .date_started')[i].text.split('\n')[3]
            date_started.append(started)

            ended = soup.select('.bookalike .date_read')[i].text.split('\n')[3]
            date_completed.append(ended)

        #Next page
        p=p+1

    #Creating the DataFrame
    index = 'Book Name, Author, ISBN 13, ISBN, Number of Pages, Average Rating, Total Number of Ratings, Publication Date, Date Started, Date Completed'.split(', ')
    book_df = pd.DataFrame(data = [books, authors, isbn13s, isbns, pagess, ratings, num_ratings, pub_dates, date_started, date_completed], index = index).transpose()
    return book_df